In [ ]:
# Original computation environment: Google Colab GPU runtime.
import os
import numpy as np
import pandas as pd
import transformers as tf

os.makedirs('../outputs', exist_ok=True)


In [ ]:
# Run this notebook from its notebooks/ directory in a Google Colab GPU runtime.
# Upload or clone the repository into the Colab session; no personal Drive mount is required.


## Prompt

> See [here](https://www.reddit.com/r/WritingPrompts/wiki/rules/)
> - Rule 1: Direct prompt replies must be good-faith attempts at new stories or poems
> - Rule 2: No explicitly sexual responses, hate speech, or other harmful content

In [ ]:
original_data = pd.read_csv('../data/external/hanna_subset.csv').rename(columns={
    'story_id': 'Story ID', 'prompt_id': 'Prompt ID', 'prompt': 'Prompt',
    'story': 'Story', 'model': 'Model', 'word_count': 'Word Count',
    'wc_diff': 'WC Diff',
})

original_data


In [ ]:
def get_writing_instruction(writing_prompt, length):
    writing_instruct = f"""
    Writing prompt:\n{writing_prompt}

    Based on the given prompt, you should create an original story, following the rules below:
    1. Stories must be good-faith attempts at new stories.
    2. Stories must be in English. No other languages, strings of binary, emoji, wingdings, etc.
    3. No explicitly sexual responses, hate speech, or other harmful content.

    Your story should contain approximately {length*.9:.0f} to {length*1.1:.0f} words. Please respond with only the story, without any additional information.
    """
    writing_instruct = writing_instruct.replace('    ','')
    return writing_instruct

print(get_writing_instruction('this is a testing prompt', 100))


## Model

### Commercial Model

In [ ]:
# models = ['gpt-5-ca', 'gpt-5-mini-ca', 'gpt-5-nano-ca']

# Unknown parameter size


### Open Model

In [ ]:
# model list: 'Qwen/Qwen2.5-7B-Instruct', 'tiiuae/Falcon3-7B-Instruct', 'google/gemma-2-9b-it'
# see: https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard#/?types=chat%2Cpretrained%2Ccontinuously+pretrained%2Cfine-tuned%2Cmerge&params=1%2C10&precision=bfloat16%2Cfloat16&filters=is_not_available_on_hub%2Cis_flagged%2Cis_merged%2Cis_moe&official=true

# We selected these models for efficiency and comparability, which are
# (1) runnable with less resources,
# (2) small enough, not flagship version, at least comparable to GPT-2 and
# (3) yielded rather satisfactory performance, esp. on instruction-following.

models = ['Qwen/Qwen2.5-7B-Instruct', 'tiiuae/Falcon3-7B-Instruct', 'google/gemma-2-9b-it']
model_id = ['-'.join(m.split('/')[1].split('-')[:2]) for m in models]

print(model_id)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import pandas as pd
from tqdm import tqdm

# 配置
MODELS = [
    'Qwen/Qwen2.5-7B-Instruct',
    'tiiuae/Falcon3-7B-Instruct',
    'google/gemma-2-9b-it'
]

OUTPUT_FILE = "../outputs/369-gen-data.json"
NUM_RUNS = 3
BATCH_SIZE = 8

# 假设您的 DataFrame 名为 df，包含 prompts 的列名为 'prompt_column'
df = original_data[original_data['Model'] == 'Human']

def cleanup_memory():
    """清理显存"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def load_results():
    """加载已有结果"""
    try:
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    except:
        return {}

def save_results(results):
    """保存结果"""
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

def generate_batch(model, tokenizer, prompts, model_name):
    """批量生成文本"""
    try:
        # 准备批量输入
        batch_texts = []
        for prompt in prompts:
            if "Qwen" in model_name:
                messages = [{"role": "user", "content": prompt}]
                text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            elif "Falcon" in model_name:
                text = f"User: {prompt}\nAssistant:"
            elif "gemma" in model_name:
                messages = [{"role": "user", "content": prompt}]
                text = tokenizer.apply_chat_template(messages, tokenize=False)
            else:
                text = prompt
            batch_texts.append(text)

        # 批量编码
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=2048)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # 批量生成
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=800,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )

        # 批量解码
        responses = []
        for i, output in enumerate(outputs):
            full_text = tokenizer.decode(output, skip_special_tokens=True)
            response = full_text.replace(batch_texts[i], "").strip()
            responses.append(response)

        return responses

    except Exception as e:
        return [f"ERROR: {str(e)}"] * len(prompts)

# 主执行逻辑
results = load_results()

# 从 DataFrame 获取 prompts
prompts = []
prompt_ids = []
for idx, row in df.iterrows():
    prompts.append(get_writing_instruction(row['Prompt'], row['Word Count']))
    prompt_ids.append(row['Prompt ID'])

print(f"找到 {len(prompts)} 个 prompts")

# 确保在 GPU 上运行
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

for model_name in MODELS:
    if model_name not in results:
        results[model_name] = {}

    print(f"\n开始处理模型: {model_name}")

    # 加载模型到 GPU
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if "Qwen" in model_name:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    ).to(device)  # 显式移动到 GPU

    # 处理每个 prompt 的多次运行
    for run_id in range(NUM_RUNS):
        print(f"  运行第 {run_id + 1} 轮")

        # 批量处理 prompts
        for batch_start in tqdm(range(0, len(prompts), BATCH_SIZE), desc=f"Run {run_id+1}"):
            batch_end = min(batch_start + BATCH_SIZE, len(prompts))
            batch_prompts = prompts[batch_start:batch_end]
            batch_ids = prompt_ids[batch_start:batch_end]

            # 检查哪些还需要运行
            need_run_indices = []
            need_run_prompts = []
            need_run_ids = []

            for i, prompt_id in enumerate(batch_ids):
                if prompt_id not in results[model_name]:
                    results[model_name][prompt_id] = []

                # 如果当前运行次数还不够，则加入批次
                if len(results[model_name][prompt_id]) <= run_id:
                    need_run_indices.append(i)
                    need_run_prompts.append(batch_prompts[i])
                    need_run_ids.append(prompt_id)

            if not need_run_prompts:
                continue

            # 批量生成
            batch_responses = generate_batch(model, tokenizer, need_run_prompts, model_name)

            # 保存结果
            for i, response in enumerate(batch_responses):
                prompt_id = need_run_ids[i]
                original_index = need_run_indices[i]

                # 确保列表长度足够
                while len(results[model_name][prompt_id]) <= run_id:
                    results[model_name][prompt_id].append({})

                results[model_name][prompt_id][run_id] = {
                    "run_id": run_id,
                    "response": response
                }

            # 实时保存
            save_results(results)

            # 小批量清理
            cleanup_memory()

    # 完成一个模型后彻底清理
    del model, tokenizer
    cleanup_memory()
    print(f"完成模型 {model_name}")

print(f"\n所有任务完成！结果保存至: {OUTPUT_FILE}")
